# Traffic Sentinel — x86 + T4 install and timing check

Our development box is ARM64 + Blackwell; the organisers run **x86 + one GPU, offline**.
This notebook checks, on a Colab / Kaggle **T4** runtime (Runtime → Change runtime type → T4 GPU):

1. `pip install -r requirements.txt` resolves on x86 and the torch wheel supports the T4 (sm_75);
2. `weights/download.sh` verifies the committed weights;
3. the fp16 inference path works on the T4 (no bf16/FP8/TensorRT is used — this confirms it) and agrees with fp32;
4. `run_submission.py` finishes one sample video inside the official budget (3 × duration), network off if the sandbox allows.

Fill in the two settings in the next cell, then *Run all*. The last cell prints a summary to paste back.

In [ ]:
# ---- settings ---------------------------------------------------------------------------------
REPO = "github.com/<owner>/traffic-sentinel"   # private repo: add a Colab secret / Kaggle secret GH_TOKEN (read-only PAT)
VIDEO_DRIVE_ID = ""                             # Google Drive file id of one sample video (C3905 is the shortest, 128 s)
VIDEO_PATH = ""                                 # ...or a path to an already-available .MP4 (e.g. on mounted Drive)

In [ ]:
import os, platform, subprocess, time, json
print(platform.machine(), platform.python_version())
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv
!nproc; grep -m1 "model name" /proc/cpuinfo

In [ ]:
token = os.environ.get("GH_TOKEN")
if token is None:
    try:
        from google.colab import userdata; token = userdata.get("GH_TOKEN")
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("GH_TOKEN")
        except Exception:
            token = None
url = f"https://{token}@{REPO}.git" if token else f"https://{REPO}.git"
!rm -rf /content/ts && git clone -q --depth 1 {url} /content/ts
%cd /content/ts
!git log --oneline -1

In [ ]:
t0 = time.perf_counter()
!pip install -q -r requirements.txt
install_sec = time.perf_counter() - t0
print(f"install took {install_sec:.0f}s")

In [ ]:
import torch, ultralytics
arch = torch.cuda.get_arch_list()
cap = torch.cuda.get_device_capability(0)
print("torch", torch.__version__, "cuda", torch.version.cuda, "ultralytics", ultralytics.__version__)
print("device", torch.cuda.get_device_name(0), "capability", cap, "wheel archs", arch)
sm_ok = f"sm_{cap[0]}{cap[1]}" in arch or any(a.startswith("compute_") and int(a.split("_")[1]) <= cap[0]*10+cap[1] for a in arch)
print("T4 supported by this torch wheel:", sm_ok)
print("bf16 supported on this GPU:", torch.cuda.is_bf16_supported())

In [ ]:
!bash weights/download.sh

In [ ]:
# fetch one sample video
os.makedirs("/content/vids", exist_ok=True)
if VIDEO_PATH:
    !cp "{VIDEO_PATH}" /content/vids/
elif VIDEO_DRIVE_ID:
    !pip install -q gdown && gdown -q --id {VIDEO_DRIVE_ID} -O /content/vids/sample.MP4
!ls -la /content/vids

In [ ]:
# fp16 (what the pipeline uses on CUDA) vs fp32 on real frames: same boxes, T4-safe kernels
import sys, glob, cv2, numpy as np
sys.path.insert(0, "src")
from sentinel.detector import Detector
from sentinel.video import resize_to_work
cap_ = cv2.VideoCapture(sorted(glob.glob("/content/vids/*"))[0])
frames = []
for k in range(0, 300, 60):
    cap_.set(cv2.CAP_PROP_POS_FRAMES, k); ok, f = cap_.read()
    if ok: frames.append(resize_to_work(f, 1280 / f.shape[1]))
det16 = Detector(); d16 = det16(frames)
det32 = Detector(); det32.half = False; d32 = det32(frames)
n16, n32 = sum(len(d.conf) for d in d16), sum(len(d.conf) for d in d32)
print(f"fp16 boxes {n16}, fp32 boxes {n32}, relative diff {abs(n16-n32)/max(n32,1):.1%}")

In [ ]:
# the official harness, network off when the sandbox permits (unshare), else online with a warning
os.environ["YOLO_OFFLINE"] = "1"
cmd = "python run_submission.py --videos /content/vids --out /content/pred_t4.json"
offline = subprocess.run("unshare -rn true", shell=True).returncode == 0
if offline:
    cmd = "unshare -rn " + cmd
else:
    print("WARNING: cannot drop the network here; running online (still catches install/CUDA/timing issues)")
t0 = time.perf_counter()
rc = subprocess.run(cmd, shell=True).returncode
run_sec = time.perf_counter() - t0
!python evaluate.py --pred /content/pred_t4.json --validate-only

In [ ]:
pred = json.load(open("/content/pred_t4.json"))
summary = {"machine": platform.machine(), "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
           "cuda": torch.version.cuda, "t4_arch_ok": sm_ok, "install_sec": round(install_sec),
           "fp16_vs_fp32_box_diff": f"{abs(n16-n32)/max(n32,1):.1%}", "offline": offline, "harness_rc": rc,
           "wall_sec": round(run_sec), "per_video": pred.get("log", {})}
print(json.dumps(summary, indent=1))